In [ ]:
!pip install pandas sqlalchemy psycopg2-binary

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 30.8 MB/s eta 0:00:00


In [ ]:
import getpass
from sqlalchemy import create_engine

db_password = getpass.getpass("Neon Veritabanı Şifrenizi Yapıştırın: ")

DB_USER = "neondb_owner"
DB_HOST = "ep-tiny-silence-anpw2tr3.c-6.us-east-1.aws.neon.tech"
DB_NAME = "neondb"

connection_uri = f"postgresql://{DB_USER}:{db_password}@{DB_HOST}/{DB_NAME}?sslmode=require"

try:
    engine = create_engine(connection_uri)
    with engine.connect() as connection:
        print("✅ Başarılı! Bulut veritabanına güvenli bir şekilde bağlandın.")
except Exception as e:
    print(f"❌ Bağlantı hatası: {e}")

Neon Veritabanı Şifrenizi Yapıştırın: ··········
✅ Başarılı! Bulut veritabanına güvenli bir şekilde bağlandın.


In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/csse_covid_19_time_series/time_series_covid19_confirmed_global.csv"

df = pd.read_csv(url)
display(df.head())
df.info()

,Province/State,Country/Region,Lat,Long,1/22/20,1/23/20,1/24/20,1/25/20,1/26/20,1/27/20,...,2/28/23,3/1/23,3/2/23,3/3/23,3/4/23,3/5/23,3/6/23,3/7/23,3/8/23,3/9/23
0,NaN,Afghanistan,33.93911,67.709953,0,0,0,0,0,0,...,209322,209340,209358,209362,209369,209390,209406,209436,209451,209451
1,NaN,Albania,41.15330,20.168300,0,0,0,0,0,0,...,334391,334408,334408,334427,334427,334427,334427,334427,334443,334457
2,NaN,Algeria,28.03390,1.659600,0,0,0,0,0,0,...,271441,271448,271463,271469,271469,271477,271477,271490,271494,271496
3,NaN,Andorra,42.50630,1.521800,0,0,0,0,0,0,...,47866,47875,47875,47875,47875,47875,47875,47875,47890,47890
4,NaN,Angola,-11.20270,17.873900,0,0,0,0,0,0,...,105255,105277,105277,105277,105277,105277,105277,105277,105288,105288


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 289 entries, 0 to 288
Columns: 1147 entries, Province/State to 3/9/23
dtypes: float64(2), int64(1143), object(2)
memory usage: 2.5+ MB


In [60]:
# melt fonksiyonuyla wide --> long
new_df = df.melt(id_vars=['Province/State', 'Country/Region', 'Lat', 'Long'], var_name="Date", value_name="Cases")

In [62]:
#tekrar kontrol
display(new_df.head())
new_df.info()

,Province/State,Country/Region,Lat,Long,Date,Cases
0,NaN,Afghanistan,33.93911,67.709953,1/22/20,0
1,NaN,Albania,41.15330,20.168300,1/22/20,0
2,NaN,Algeria,28.03390,1.659600,1/22/20,0
3,NaN,Andorra,42.50630,1.521800,1/22/20,0
4,NaN,Angola,-11.20270,17.873900,1/22/20,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 330327 entries, 0 to 330326
Data columns (total 6 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   Province/State  104013 non-null  object 
 1   Country/Region  330327 non-null  object 
 2   Lat             328041 non-null  float64
 3   Long            328041 non-null  float64
 4   Date            330327 non-null  object 
 5   Cases           330327 non-null  int64  
dtypes: float64(2), int64(1), object(3)
memory usage: 15.1+ MB


In [ ]:
# object --> datetime64[ns]
new_df['Date'] = pd.to_datetime(new_df['Date'])
print(new_df.dtypes)

Province/State            object
Country/Region            object
Lat                      float64
Long                     float64
Date              datetime64[ns]
Cases                      int64
dtype: object


In [56]:
new_df.loc[new_df["Cases"] < 0]

,Province/State,Country/Region,Lat,Long,Date,Cases


In [ ]:
#İsimlendirme kontrolü
df_long['Country/Region'].unique()

array(['Afghanistan', 'Albania', 'Algeria', 'Andorra', 'Angola',
       'Antarctica', 'Antigua and Barbuda', 'Argentina', 'Armenia',
       'Australia', 'Austria', 'Azerbaijan', 'Bahamas', 'Bahrain',
       'Bangladesh', 'Barbados', 'Belarus', 'Belgium', 'Belize', 'Benin',
       'Bhutan', 'Bolivia', 'Bosnia and Herzegovina', 'Botswana',
       'Brazil', 'Brunei', 'Bulgaria', 'Burkina Faso', 'Burma', 'Burundi',
       'Cabo Verde', 'Cambodia', 'Cameroon', 'Canada',
       'Central African Republic', 'Chad', 'Chile', 'China', 'Colombia',
       'Comoros', 'Congo (Brazzaville)', 'Congo (Kinshasa)', 'Costa Rica',
       "Cote d'Ivoire", 'Croatia', 'Cuba', 'Cyprus', 'Czechia', 'Denmark',
       'Diamond Princess', 'Djibouti', 'Dominica', 'Dominican Republic',
       'Ecuador', 'Egypt', 'El Salvador', 'Equatorial Guinea', 'Eritrea',
       'Estonia', 'Eswatini', 'Ethiopia', 'Fiji', 'Finland', 'France',
       'Gabon', 'Gambia', 'Georgia', 'Germany', 'Ghana', 'Greece',
       'Grenada', 'Gua

In [ ]:
# Hatalı isimlendirmeler ve doğruları -key:value- dict
new_names = {
    'US': 'United States',
    'Korea, North' : 'North Korea',
    'Korea, South': 'South Korea',
    'Taiwan*': 'Taiwan'
}

# replace() ile eşleşen tüm isimleri tek seferde değiştir
new_df['Country/Region'] = new_df['Country/Region'].replace(new_names)

# kotrol
wrong_name = new_df.loc[new_df['Country/Region'] == 'US']
print(len(wrong_name))

0


In [ ]:
#ülke olmayan kayıtların listesi
ulke_olmayan = ['Diamond Princess', 'MS Zaandam','Summer Olympics 2020', 'Winter Olympics 2022']

# ~ işareti ile "bu listede OLMAYANLARI getir ve tablomu güncelle" diyoruz
new_df= new_df[~new_df['Country/Region'].isin(ulke_olmayan)]

gone = new_df.loc[new_df['Country/Region'] == 'Diamond Princess']
print(gone)

Empty DataFrame
Columns: [Province/State, Country/Region, Lat, Long, Date, Cases]
Index: []


In [68]:
# Temizlenmiş veriyi 'fact_covid_cases' adında bir tablo olarak veritabanına yazıyoruz
new_df.to_sql('fact_covid_cases',
               engine,
               if_exists='replace', # Tablo zaten varsa siler, yenisini oluşturur
               index=False)         # Pandas'ın sol taraftaki kendi sıra numaralarını DB'ye kaydetmesini önler

print("Veri bulut PostgreSQL'e yüklendi!")

Veri bulut PostgreSQL'e yüklendi!


In [69]:
# Doğrulama (Validation) kısmı
kontrol_df = pd.read_sql("SELECT * FROM fact_covid_cases LIMIT 5", engine)

display(kontrol_df)

,Province/State,Country/Region,Lat,Long,Date,Cases
0,None,Afghanistan,33.93911,67.709953,1/22/20,0
1,None,Albania,41.15330,20.168300,1/22/20,0
2,None,Algeria,28.03390,1.659600,1/22/20,0
3,None,Andorra,42.50630,1.521800,1/22/20,0
4,None,Angola,-11.20270,17.873900,1/22/20,0


In [ ]:
#----------------------------------------------------------------------------

In [59]:
!pip freeze

absl-py==1.4.0
accelerate==1.13.0
access==1.1.10.post3
affine==2.4.0
aiofiles==24.1.0
aiohappyeyeballs==2.6.1
aiohttp==3.13.4
aiosignal==1.4.0
aiosqlite==0.22.1
alabaster==1.0.0
albucore==0.0.24
albumentations==2.0.8
ale-py==0.11.2
alembic==1.18.4
altair==5.5.0
annotated-doc==0.0.4
annotated-types==0.7.0
antlr4-python3-runtime==4.9.3
anyio==4.13.0
anywidget==0.9.21
apsw==3.51.3.0
apswutils==0.1.2
argon2-cffi==25.1.0
argon2-cffi-bindings==25.1.0
array_record==0.8.3
arrow==1.4.0
arviz==0.22.0
astropy==7.2.0
astropy-iers-data==0.2026.3.30.0.54.34
astunparse==1.6.3
atpublic==5.1
attrs==26.1.0
audioread==3.1.0
Authlib==1.6.9
autograd==1.8.0
babel==2.18.0
backcall==0.2.0
beartype==0.22.9
beautifulsoup4==4.13.5
betterproto==2.0.0b6
bigframes==2.38.0
bigquery-magics==0.12.2
bleach==6.3.0
blinker==1.9.0
blis==1.3.3
blobfile==3.2.0
blosc2==4.1.2
bokeh==3.8.2
Bottleneck==1.4.2
bqplot==0.12.45
branca==0.8.2
brotli==1.2.0
CacheControl==0.14.4
cachetools==6.2.6
catalogue==2.0.10
certifi==2026.2.25
c

In [ ]:
new_df.sort_values(["Country/Region"], ascending= False)

,Country/Region,Lat,Long,Date,Cases
330615,Zimbabwe,-19.015438,29.154857,3/9/23,264276
184959,Zimbabwe,-19.015438,29.154857,10/21/21,132540
277728,Zimbabwe,-19.015438,29.154857,9/7/22,256825
326280,Zimbabwe,-19.015438,29.154857,2/22/23,263921
55776,Zimbabwe,-19.015438,29.154857,7/31/20,3169
...,...,...,...,...,...
218195,Afghanistan,33.939110,67.709953,2/14/22,171246
217906,Afghanistan,33.939110,67.709953,2/13/22,170604
217617,Afghanistan,33.939110,67.709953,2/12/22,170152
217328,Afghanistan,33.939110,67.709953,2/11/22,169940


In [ ]:
# melt() fonksiyonu örnek

cheese = pd.DataFrame(
    {
        "first": ["John", "Mary"],
        "last": ["Doe", "Bo"],
        "height": [5.5, 6.0],
        "weight": [130, 150],
    }
)

cheese.melt(id_vars = ['first', 'last'], var_name="quantity", value_name="brm")

--------------------------------


,first,last,quantity,brm
0,John,Doe,height,5.5
1,Mary,Bo,height,6.0
2,John,Doe,weight,130.0
3,Mary,Bo,weight,150.0
